In [2]:
import napari, mrcfile, tifffile as tiff, numpy as np
from pathlib import Path

mrc_path = next(Path(r"C:\Users\pauum\Desktop\PAULA\uni\MASTER\TFM\CRYOEM\Proyecto\datos").glob("*.mrc"))
with mrcfile.mmap(mrc_path, permissive=True) as m:
    tomo = np.asarray(m.data)
    vox = m.voxel_size
print(tomo.shape, tomo.dtype)

viewer = napari.Viewer(ndisplay=3)     # arranca ya en modo 3D
lo, hi = np.percentile(tomo[::4, ::4, ::4], (1, 99))   # percentiles sobre una versión submuestreada (más rápido)
viewer.add_image(tomo, name="tomograma", contrast_limits=(lo, hi),
                 colormap="gray", rendering="attenuated_mip")

# Anotaciones como capas de etiquetas (una por archivo)
for f in sorted(Path(r"C:\Users\pauum\Desktop\PAULA\uni\MASTER\TFM\CRYOEM\Proyecto\datos").glob("annotations*.tif")):
    a = tiff.imread(f)
    if a.shape != tomo.shape:
        print(f"{f.name}: shape {a.shape} ≠ tomograma {tomo.shape}, la salto")
        continue
    viewer.add_labels((a < a.max() - 1e-6).astype(np.uint8), name=f.stem, opacity=0.5)

napari.run()

(200, 512, 512) float32
